In [3]:
import torch
import os

# --- FIX: FORCE INITIALIZATION FOR TORCH._DYNAMO ---
try:
    import torch._dynamo
    import torch._dynamo.decorators
except (ImportError, AttributeError):
    pass
# --------------------------------------------------

import gymnasium as gym
from gymnasium import spaces
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.results_plotter import load_results, ts2xy
import matplotlib.pyplot as plt
import pandas as pd
import random
from collections import deque

class MahimahiTraceManager:
    """
    Mengelola file trace Mahimahi (.log).
    Mengonversi timestamp (ms) menjadi Throughput (Mbps) dengan amplifikasi yang realistis.
    """
    def __init__(self, folder_path="traces_folder"):
        self.traces = []
        PACKET_SIZE_BITS = 1500 * 8 
        
        if os.path.exists(folder_path):
            files = [f for f in os.listdir(folder_path) if f.endswith('.log')]
            files.sort()
            
            for file in files:
                path = os.path.join(folder_path, file)
                try:
                    with open(path, 'r') as f:
                        timestamps_ms = [float(line.strip()) for line in f if line.strip()]
                        
                        if timestamps_ms:
                            throughput_mbps = []
                            current_sec = 0
                            packet_count = 0
                            
                            for ts in timestamps_ms:
                                sec = int(ts / 1000)
                                while current_sec < sec:
                                    mbps = (packet_count * PACKET_SIZE_BITS) / 1_000_000
                                    throughput_mbps.append(mbps)
                                    packet_count = 0
                                    current_sec += 1
                                packet_count += 1
                            
                            throughput_mbps.append((packet_count * PACKET_SIZE_BITS) / 1_000_000)
                            
                            # Scaling Realistis (Max ~12 Mbps)
                            max_tp = max(throughput_mbps) if throughput_mbps else 1
                            scale_factor = 12.0 / max_tp if max_tp > 0 else 1.0
                            
                            scaled_mbps = [max(0.1, tp * scale_factor) for tp in throughput_mbps]
                            # Pre-smoothing trace
                            smoothed = pd.Series(scaled_mbps).rolling(window=3, min_periods=1).mean().tolist()
                            
                            self.traces.append({
                                "name": file,
                                "data": smoothed
                            })
                except Exception as e:
                    print(f"Gagal memproses file {file}: {e}")
        
        if not self.traces:
            print("⚠️ Folder traces_folder tidak ditemukan atau kosong!")
            self.traces.append({"name": "synth_fallback", "data": [8.0] * 500})

        self.active_trace = None
        self.ptr = 0

    def select_random_trace(self):
        self.active_trace = random.choice(self.traces)
        self.ptr = random.randint(0, max(0, len(self.active_trace["data"]) - 110))
        return self.active_trace["name"]

    def set_trace_index(self, idx):
        self.active_trace = self.traces[idx % len(self.traces)]
        self.ptr = 0
        return self.active_trace["name"]

    def get_next_bandwidth(self):
        val = self.active_trace["data"][self.ptr]
        self.ptr = (self.ptr + 1) % len(self.active_trace["data"])
        return val

class HybridStreamingEnv(gym.Env):
    """
    Arsitektur Hybrid Terbaru: RL (Strategic) + Improved Deterministic Controller.
    """
    def __init__(self, trace_manager):
        super(HybridStreamingEnv, self).__init__()
        self.trace_manager = trace_manager
        self.bitrates = [0.5, 2.5, 8.0] 
        
        # PERBAIKAN 1: Observasi yang lebih bersih dan stabil
        # [Buffer_Norm, Smoothed_TP_Norm, Last_Exec, Buffer_Safety_Ratio]
        self.observation_space = spaces.Box(
            low=np.array([0.0, 0.0, 0.0, 0.0]),
            high=np.array([1.0, 1.0, 1.0, 1.0]), 
            dtype=np.float32
        )
        
        self.action_space = spaces.Discrete(3)
        
        self.state = None 
        self.max_steps = 100
        self.current_step = 0
        
        # PERBAIKAN 2: Perpanjang history untuk smoothing yang lebih baik
        self.tp_history = deque(maxlen=10) 
        
        # Controller State
        self.current_exec_idx = 1
        self.upgrade_confidence_counter = 0
        
        # Parameter Controller yang Disesuaikan
        self.LOW_BUFFER_THRESHOLD = 5.0 
        self.SAFE_MARGIN = 1.5         
        self.UPGRADE_CONFIRM_STEPS = 2
        self.PANIC_BUFFER_THRESHOLD = 3.0

    def _get_normalized_obs(self):
        # State: [buffer, smoothed_tp, last_exec, buf_trend, raw_tp]
        buffer, smoothed_tp, last_exec, _, _ = self.state
        
        norm_buffer = np.clip(buffer / 30.0, 0.0, 1.0)           
        
        # Ratio: Throughput / Highest Bitrate logis
        norm_tp = np.clip(smoothed_tp / 10.0, 0.0, 1.0) 
        
        norm_exec = float(last_exec) / 2.0                   
        
        # Fitur baru: Buffer Safety
        safety = max(0, buffer - self.LOW_BUFFER_THRESHOLD)
        norm_safety = np.clip(safety / 15.0, 0.0, 1.0) 
        
        return np.array([norm_buffer, norm_tp, norm_exec, norm_safety], dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if options and "trace_idx" in options:
            trace_name = self.trace_manager.set_trace_index(options["trace_idx"])
        else:
            trace_name = self.trace_manager.select_random_trace()
            
        initial_tp = self.trace_manager.get_next_bandwidth()
        
        self.tp_history.clear()
        self.upgrade_confidence_counter = 0
        self.current_exec_idx = 1
        
        for _ in range(self.tp_history.maxlen):
            self.tp_history.append(initial_tp)
            
        # state: [buffer, smoothed_tp, last_exec, buf_trend, raw_tp]
        self.state = np.array([10.0, initial_tp, 1.0, 0.0, initial_tp], dtype=np.float32)
        self.current_step = 0
        return self._get_normalized_obs(), {"trace": trace_name}

    def step(self, action):
        if isinstance(action, np.ndarray):
            action = action.item()
        
        target_idx = int(action)
        buffer, _, current_idx, _, _ = self.state
        current_idx = int(current_idx)
        
        # 1. Dapatkan Throughput
        raw_tp = self.trace_manager.get_next_bandwidth()
        self.tp_history.append(raw_tp)
        
        # PERBAIKAN 3: Moving Average TP
        smoothed_tp = sum(self.tp_history) / len(self.tp_history)
        
        # Hitung Headroom berdasarkan SMOOTHED TP
        headroom = smoothed_tp - self.bitrates[current_idx]
        
        # --- LAYER 2: IMPROVED SMOOTHING CONTROLLER ---
        executed_idx = current_idx

        # 1. Rate Limiter (Max 1 level per step)
        if abs(target_idx - current_idx) > 1:
            target_idx = current_idx + np.sign(target_idx - current_idx)

        # 2. PANIC Mode (Hanya turun jika buffer BENAR-BENAR KRITIS)
        if buffer < self.PANIC_BUFFER_THRESHOLD:
            executed_idx = min(target_idx, current_idx - 1)
            executed_idx = max(0, executed_idx)
            self.upgrade_confidence_counter = 0
            
        # 3. Defensive Downgrade (Hanya jika Headroom negatif SIGNIFIKAN dan buffer rendah)
        elif headroom < -2.0 and buffer < self.LOW_BUFFER_THRESHOLD:
            executed_idx = max(0, current_idx - 1)
            self.upgrade_confidence_counter = 0

        # 4. Upgrade Hysteresis (Dengan kondisi buffer aman)
        elif target_idx > current_idx:
            if headroom > self.SAFE_MARGIN and buffer > self.LOW_BUFFER_THRESHOLD:
                self.upgrade_confidence_counter += 1
            else:
                self.upgrade_confidence_counter = 0
            
            if self.upgrade_confidence_counter >= self.UPGRADE_CONFIRM_STEPS:
                executed_idx = current_idx + 1
                self.upgrade_confidence_counter = 0
            else:
                executed_idx = current_idx 
                
        # 5. Normal Action (Jika target == current atau target < current)
        else:
            executed_idx = target_idx
            self.upgrade_confidence_counter = 0

        executed_idx = np.clip(executed_idx, 0, 2)

        # Eksekusi Fisik
        chosen_bitrate = self.bitrates[executed_idx]
        seg_duration = 5.0
        
        # Kalkulasi realitas pakai raw_tp, keputusan pakai smoothed
        download_time = (chosen_bitrate * seg_duration / (raw_tp + 0.1)) 
        stalling = max(0, download_time - buffer)
        
        new_buffer = max(0, buffer - download_time) + seg_duration
        new_buffer = min(new_buffer, 30.0)
        
        buf_trend = new_buffer - buffer

        # --- REWARD STRATEGI ---
        reward = float(chosen_bitrate * 1.0)
        
        if stalling > 0:
            reward -= float(stalling * 50.0 + 20.0) 
            
        # PERBAIKAN 4: Penalti Switch yang adaptif
        if executed_idx != current_idx:
            if buffer > 15.0: # Switch saat buffer aman? (Mencegah Jitter)
                reward -= 5.0 
            else:
                reward -= 1.0 # Penalti normal untuk adaptasi

        # Update Internal State
        self.state = np.array([new_buffer, smoothed_tp, float(executed_idx), buf_trend, raw_tp], dtype=np.float32)
        self.current_step += 1
        done = self.current_step >= self.max_steps
        
        info = {
            "raw_tp": raw_tp, 
            "buffer": new_buffer,
            "executed_bitrate": chosen_bitrate,
            "smoothed_tp": smoothed_tp,
            "target_idx": target_idx,
            "executed_idx": executed_idx
        }
        
        return self._get_normalized_obs(), reward, done, False, info

def run_experiment():
    log_dir = "./rl_logs/"
    if not os.path.exists(log_dir):
        os.makedirs(log_dir)
    
    tm = MahimahiTraceManager(folder_path="traces_folder/mahimahi")
    if not tm.traces:
        print("❌ Tidak ada file trace yang ditemukan.")
        return
        
    env = Monitor(HybridStreamingEnv(tm), log_dir)
    
    print("⏳ Memulai pelatihan agen 'Intelligent-Hybrid ABR' (300,000 langkah)...")
    model = PPO("MlpPolicy", env, verbose=1, 
                learning_rate=0.00025,  
                ent_coef=0.05, 
                n_steps=2048,
                batch_size=64)         
    
    model.learn(total_timesteps=300000)
    model.save("intelligent_hybrid_ndn_model")
    print("✅ Pelatihan selesai. Model disimpan.")

    print("\n📈 Menghasilkan 8 Grafik Evaluasi (Improved Hybrid)...")
    for i in range(len(tm.traces)):
        obs, info = env.reset(options={"trace_idx": i})
        trace_name = info["trace"]
        history = []
        
        for _ in range(100):
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, _, info_step = env.step(action)
            
            history.append({
                'Throughput': info_step['raw_tp'],
                'Smoothed_TP': info_step['smoothed_tp'],
                'Buffer': info_step['buffer'], 
                'Exec_Bitrate': info_step['executed_bitrate'],
                'Target_Idx': info_step['target_idx']
            })
            
        df = pd.DataFrame(history)
        
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
        
        # Grafik Bitrate
        ax1.plot(df.index, df['Throughput'], label='Raw Bandwidth', color='blue', alpha=0.15)
        ax1.plot(df.index, df['Smoothed_TP'], label='Smoothed Bandwidth', color='blue', alpha=0.5, linestyle='--')
        ax1.step(df.index, df['Exec_Bitrate'], label='Executed Bitrate', color='red', linewidth=2.5, where='post')
        
        ax1.set_title(f"Evaluasi Improved Hybrid: {trace_name}")
        ax1.set_ylabel("Mbps")
        ax1.legend()
        ax1.grid(alpha=0.2)
        
        # Grafik Buffer
        ax2.fill_between(df.index, df['Buffer'], color='green', alpha=0.15, label='Buffer Level')
        ax2.axhline(y=3, color='red', linestyle='--', label='Panic Zone (3s)')
        ax2.axhline(y=5, color='orange', linestyle='--', label='Low Buffer (5s)')
        ax2.set_ylabel("Detik")
        ax2.set_xlabel("Segmen")
        ax2.set_ylim(0, 35)
        ax2.legend()
        ax2.grid(alpha=0.2)
        
        plt.tight_layout()
        plot_filename = os.path.join(log_dir, f"improved_hybrid_{trace_name}.png")
        plt.savefig(plot_filename)
        plt.close()

    print(f"🎉 Selesai! Cek folder {log_dir} untuk hasil grafiknya.")

if __name__ == "__main__":
    run_experiment()

⏳ Memulai pelatihan agen 'Intelligent-Hybrid ABR' (300,000 langkah)...
Using cpu device
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 100      |
|    ep_rew_mean     | -62.4    |
| time/              |          |
|    fps             | 4553     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 100         |
|    ep_rew_mean          | -38         |
| time/                   |             |
|    fps                  | 3010        |
|    iterations           | 2           |
|    time_elapsed         | 1           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.009557152 |
|    clip_fraction        | 0.103       |
|    clip_range           | 0.2  